In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib

In [ ]:
df = pd.read_parquet("df_final.parquet")

df.head()

,_ASTHMS1,CHCKDNY2,_DRDXAR2,_EDUCAG,_INCOMG1,_AGE_G,_SEX,WTKG3,HTM4,_RFBMI5,_BMI5,_RFDRHV9,_RFSMOK3,_TOTINDA,DIABETE4,year,DIABETE_BIN
0,3.0,0.0,1.0,2.0,9.0,6.0,1.0,5942.0,163.0,1.0,2249.0,1.0,1.0,1.0,3.0,2024,0.0
1,3.0,0.0,1.0,4.0,7.0,6.0,0.0,8165.0,178.0,2.0,2583.0,1.0,1.0,1.0,3.0,2024,0.0
2,3.0,0.0,1.0,3.0,9.0,5.0,0.0,8845.0,198.0,1.0,2253.0,1.0,0.0,1.0,3.0,2024,0.0
3,3.0,0.0,1.0,4.0,4.0,6.0,0.0,7484.0,173.0,2.0,2509.0,1.0,1.0,1.0,3.0,2024,0.0
4,3.0,0.0,0.0,3.0,2.0,4.0,0.0,5897.0,173.0,1.0,1977.0,1.0,1.0,2.0,3.0,2024,0.0


## Features
### Target
- DIABETE4

### Features (Use)
- _AGE_G
- _SEX
- _EDUCAG
- _INCOMG1
- _TOTINDA
- _RFSMOK3
- _RFDRHV9
- _BMI5
- CHCKDNY2 (bias, kidney disease)
- _DRDXAR2 (bias)
- year


### Dropping
- _RFBMI5
- WTKG3
- HTM4

In [ ]:
cols_to_drop = ["_RFBMI5", "WTKG3", "HTM4"]
df = df.drop(columns=cols_to_drop)
df.head()

,_ASTHMS1,CHCKDNY2,_DRDXAR2,_EDUCAG,_INCOMG1,_AGE_G,_SEX,_BMI5,_RFDRHV9,_RFSMOK3,_TOTINDA,DIABETE4,year,DIABETE_BIN
0,3.0,0.0,1.0,2.0,9.0,6.0,1.0,2249.0,1.0,1.0,1.0,3.0,2024,0.0
1,3.0,0.0,1.0,4.0,7.0,6.0,0.0,2583.0,1.0,1.0,1.0,3.0,2024,0.0
2,3.0,0.0,1.0,3.0,9.0,5.0,0.0,2253.0,1.0,0.0,1.0,3.0,2024,0.0
3,3.0,0.0,1.0,4.0,4.0,6.0,0.0,2509.0,1.0,1.0,1.0,3.0,2024,0.0
4,3.0,0.0,0.0,3.0,2.0,4.0,0.0,1977.0,1.0,1.0,2.0,3.0,2024,0.0


In [ ]:
df["DIABETE_BIN"].isna().sum()
df = df.dropna(subset=["DIABETE_BIN"])
df["DIABETE_BIN"].value_counts(normalize=True) * 100

,proportion
DIABETE_BIN,
0.0,83.44219
1.0,16.55781


## Simple DNN

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score


target_col = "DIABETE_BIN"
batch_size = 512
epochs = 5
lr = 0.0003

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



X = df.drop(columns=[target_col, "DIABETE4", "year"])
y = df[target_col]
X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")



class HealthDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = HealthDataset(X_train, y_train)
test_ds = HealthDataset(X_test, y_test)



train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)



class DiabetesDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


model = DiabetesDNN(input_dim=X_train.shape[1]).to(device)






pos = (y_train == 1).sum()
neg = (y_train == 0).sum()

pos_weight = torch.tensor([neg / (pos + 1e-6)], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)



for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(X_batch).view(-1)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")



model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device, non_blocking=True)

        logits = model(X_batch).view(-1)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())


all_preds = [1 if p > 0.5 else 0 for p in all_probs]


acc = accuracy_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)
f1 = f1_score(all_labels, all_preds, average="macro")

print("\n===== TEST METRICS =====")
print(f"Accuracy : {acc:.4f}")
print(f"ROC AUC  : {auc:.4f}")
print(f"F1 Macro : {f1:.4f}")




Using device: cpu


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to

Epoch 1/5 | Loss: 0.9697
Epoch 2/5 | Loss: 0.9573
Epoch 3/5 | Loss: 0.9553
Epoch 4/5 | Loss: 0.9539
Epoch 5/5 | Loss: 0.9533

===== TEST METRICS =====
Accuracy : 0.6658
ROC AUC  : 0.7731
F1 Macro : 0.5961


## DNN + GPU + BALANCING DATA

In [ ]:

X = df.drop(columns=[target_col, "DIABETE4", "year"])
y = df[target_col]

X = X.fillna(X.median(numeric_only=True))


X = X.clip(X.quantile(0.01), X.quantile(0.99), axis=1)

skewed_cols = X.skew().abs() > 1
X.loc[:, skewed_cols] = np.log1p(X.loc[:, skewed_cols])



X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")


class HealthDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = HealthDataset(X_train, y_train)
test_ds = HealthDataset(X_test, y_test)


train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)



class DiabetesDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


model = DiabetesDNN(X_train.shape[1]).to(device)


pos = (y_train == 1).sum()
neg = (y_train == 0).sum()

pos_weight = torch.tensor([neg / (pos + 1e-6)], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)


scaler_amp = torch.cuda.amp.GradScaler()


for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            logits = model(X_batch).view(-1)
            loss = criterion(logits, y_batch)

        scaler_amp.scale(loss).backward()
        scaler_amp.step(optimizer)
        scaler_amp.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")



model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device, non_blocking=True)

        logits = model(X_batch).view(-1)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())



threshold = 0.5
all_preds = [1 if p > threshold else 0 for p in all_probs]



acc = accuracy_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)
f1 = f1_score(all_labels, all_preds, average="macro")

print("\n===== TEST METRICS =====")
print(f"Accuracy : {acc:.4f}")
print(f"ROC AUC  : {auc:.4f}")
print(f"F1 Macro : {f1:.4f}")



/tmp/ipykernel_29473/385769175.py:97: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_29473/385769175.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Epoch 1/5 | Loss: 0.9575
Epoch 2/5 | Loss: 0.9484
Epoch 3/5 | Loss: 0.9474
Epoch 4/5 | Loss: 0.9468
Epoch 5/5 | Loss: 0.9463

===== TEST METRICS =====
Accuracy : 0.6614
ROC AUC  : 0.7737
F1 Macro : 0.5937


In [ ]:
torch.save(model.state_dict(), "diabetes_model.pt")
print("\nModel saved.")


Model saved.


* Built a PyTorch DNN for diabetes prediction on a 1.3M-row dataset with proper preprocessing (scaling, outlier clipping, log transform).
* Fixed training instability (NaN loss) by tuning learning rate and handling class imbalance with `pos_weight`.
* Optimized pipeline using larger batch size, better DataLoader settings.
* Achieved ~0.77 ROC AUC, ~0.66 accuracy, and ~0.59 macro F1.

